In [0]:
%sql
-- Drop and create analytical view table
DROP TABLE IF EXISTS practice.bricks.car_sales_analytical_view;

CREATE TABLE practice.bricks.car_sales_analytical_view AS
SELECT 
  -- Fact table columns
  f.sales_id,
  f.selling_price,
  f.mmr,
  f.odometer,
  f.condition,
  
  -- Date dimension columns
  d.sale_date,
  d.year AS sale_year,
  d.month AS sale_month,
  d.month_name AS sale_month_name,
  d.day AS sale_day,
  d.weekday AS sale_weekday,
  
  -- Vehicle dimension columns
  v.vin,
  v.year AS vehicle_year,
  v.make,
  v.model,
  v.trim,
  v.body_style,
  v.transmission,
  v.color,
  v.interior,
  
  -- Seller dimension columns
  s.seller_name,
  
  -- Location dimension columns
  l.state
  
FROM practice.bricks.fact_sales f
INNER JOIN practice.bricks.dim_vehicle v ON f.vehicle_id = v.vehicle_id
INNER JOIN practice.bricks.dim_seller s ON f.seller_id = s.seller_id
INNER JOIN practice.bricks.dim_location l ON f.location_id = l.location_id
INNER JOIN practice.bricks.dim_date d ON f.date_id = d.date_id;

In [0]:
%sql
-- Query the analytical view - no joins needed!
SELECT 
  sales_id,
  sale_date,
  sale_year,
  sale_month_name,
  vehicle_year,
  make,
  model,
  color,
  seller_name,
  state,
  selling_price,
  mmr,
  odometer,
  condition
FROM practice.bricks.car_sales_analytical_view
LIMIT 10;

In [0]:
%sql
-- Add calculated metrics to analytical view
DROP TABLE IF EXISTS practice.bricks.car_sales_analytical_view;

CREATE TABLE practice.bricks.car_sales_analytical_view AS
SELECT 
  -- Fact table columns
  f.sales_id,
  f.selling_price,
  f.mmr,
  f.odometer,
  f.condition,
  
  -- Date dimension columns
  d.sale_date,
  d.year AS sale_year,
  d.month AS sale_month,
  d.month_name AS sale_month_name,
  d.day AS sale_day,
  d.weekday AS sale_weekday,
  
  -- Vehicle dimension columns
  v.vin,
  v.year AS vehicle_year,
  v.make,
  v.model,
  v.trim,
  v.body_style,
  v.transmission,
  v.color,
  v.interior,
  
  -- Seller dimension columns
  s.seller_name,
  
  -- Location dimension columns
  l.state,
  
  -- Calculated fields (Gold layer enhancements)
  (f.selling_price - f.mmr) AS premium_over_mmr,
  ROUND(((f.selling_price - f.mmr) / NULLIF(f.mmr, 0)) * 100, 2) AS premium_pct,
  ROUND(f.selling_price / NULLIF(f.odometer, 0), 2) AS price_per_mile,
  (d.year - v.year) AS vehicle_age,
  CASE WHEN f.selling_price > f.mmr THEN 1 ELSE 0 END AS is_above_mmr
  
FROM practice.bricks.fact_sales f
INNER JOIN practice.bricks.dim_vehicle v ON f.vehicle_id = v.vehicle_id
INNER JOIN practice.bricks.dim_seller s ON f.seller_id = s.seller_id
INNER JOIN practice.bricks.dim_location l ON f.location_id = l.location_id
INNER JOIN practice.bricks.dim_date d ON f.date_id = d.date_id;